# Consultas SQL· Cuaderno del proyecto
# Caracterización de los homicidios en Colombia durante el periodo noviembre/2025 - octubre/2026


**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [ ]:
# Celda 2 · Cargar la muestra
ENLACE = "https://github.com/eaguca/Caracterizacion-de-los-homicidios-en-Colombia-durante-el-periodo-nov-2025-oct-2026/blob/main/datos/originales/HOMICIDIO_20261123.csv"
if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

Cargué 27801 filas y 11 columnas.


,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,ZONA,SEXO,ARMA MEDIO,MODALIDAD PRESUNTA,SPOA_CARACTERIZACION,CANTIDAD
0,02/11/2024,8,ATLANTICO,8758,SOLEDAD,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1
1,02/11/2024,8,ATLANTICO,8758,SOLEDAD,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1
2,02/11/2024,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA BLANCA / CORTOPUNZANTE,ATRACO,HOMICIDIO INTENCIONAL O CON DOLO,1
3,02/11/2024,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA BLANCA / CORTOPUNZANTE,RIÑAS,HOMICIDIO INTENCIONAL O CON DOLO,1
4,02/11/2024,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA BLANCA / CORTOPUNZANTE,RIÑAS,HOMICIDIO INTENCIONAL O CON DOLO,1


## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [ ]:
# Celda 3 · Contar filas totales del dataset
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

,filas
0,27801


## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [ ]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

['FECHA HECHO',
 'COD_DEPTO',
 'DEPARTAMENTO',
 'COD_MUNI',
 'MUNICIPIO',
 'ZONA',
 'SEXO',
 'ARMA MEDIO',
 ' MODALIDAD PRESUNTA',
 'SPOA_CARACTERIZACION',
 'CANTIDAD']

---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [ ]:
# Celda 5 · Consulta 1a: filas antes del filtro (¿Cuántas filas tiene el dataset?
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

,filas_total
0,27801


In [ ]:
# Celda 6 · Consulta 1b: filas después del filtro (Filas del año 2025)
sql("""
SELECT COUNT(*) AS filas
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
""")

,filas
0,13981


In [ ]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas (Total de víctimas del 2025)
sql("""
SELECT SUM(cantidad)
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
""")

,sum(cantidad)
0,14038.0


**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [ ]:
# Celda 8 · Consulta 2: resumen por grupo (¿Cuántos homicidios occurrieron en 2025 discriminados por zona rural y urbana?)
sql("""
SELECT ZONA, SUM(CANTIDAD) AS total_homicidios
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY ZONA
""")

,ZONA,total_homicidios
0,URBANA,9310.0
1,RURAL,4728.0


In [ ]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql(consulta="""
SELECT SUM(total_homicidios) AS suma_de_los_grupos
FROM (
    SELECT ZONA, SUM(CANTIDAD) AS total_homicidios
    FROM datos
    WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
    GROUP BY ZONA
)
""")

,suma_de_los_grupos
0,14038.0


---
# Sesión 4 · Tu reto
Antes de decir cuántos homicidios hubo: ¿puedes confiar
en la base?

>**El contexto:**
>
>Una concejala de Medellín prepara un debate de control sobre seguridad y le pide a su equipo una sola cifra:
>cuántos homicidios hubo en 2025 en Medellín y cómo se reparten entre zona urbana y rural. Si la cifra está
>mal, el debate se cae en el primer minuto. Ustedes tienen la base; antes de entregar el número, tienen que
>poder decir qué cuenta y qué no saben.

**¿Filas o víctimas?**


Cuenten las filas y sumen cantidad. ¿Dan lo mismo? ¿Cuántas filas tienen una cantidad mayor que 1? Su indicador principal, ¿cuenta filas o víctimas? Escríbanlo en una frase.

In [ ]:
# Celda 10 · ¿Cuántas filas hay del año 2025?
sql("""
SELECT COUNT(*) AS filas_2025
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
""")

,filas_2025
0,13981


In [ ]:
# Celda 11 · ¿Cuánto es la suma de la columna CANTIDAD para el año 2025?
sql("""
SELECT SUM(CANTIDAD)
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
""")

,sum(CANTIDAD)
0,14038.0


In [ ]:
# Celda 12 · ¿Cuántas filas tienen una cantidad mayor que uno para el año 2025?
sql("""
SELECT COUNT(*) AS filas
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31' AND cantidad > 1
""")

,filas
0,52


# Nuestro indicador principal cuenta víctimas, es decir, suma el contenido de la columna cantidad.

**Filas idénticas: ¿copias o víctimas distintas?**

In [ ]:
# Celda 13 · ¿Cuántas filas son idénticas a otra para el año 2025?
sql(consulta="""
WITH filas_2025 AS (
    SELECT *
    FROM datos
    WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
),
conteo_grupos AS (
    SELECT *, COUNT(*) as repeticiones
    FROM filas_2025
    GROUP BY ALL
)
SELECT SUM(repeticiones) AS total_filas_duplicadas
FROM conteo_grupos
WHERE repeticiones > 1
""")

,total_filas_duplicadas
0,3660.0


In [ ]:
# Celda 14 · ¿Cuáles son las filas idénticas que más se repiten para el año 2025?
sql("""
WITH filas_2025 AS (
    SELECT *
    FROM datos
    WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
)
SELECT *, COUNT(*) as repeticiones
FROM filas_2025
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY repeticiones DESC
LIMIT 10
""")

,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,ZONA,SEXO,ARMA MEDIO,MODALIDAD PRESUNTA,SPOA_CARACTERIZACION,CANTIDAD,repeticiones
0,16/01/2025,54,NORTE DE SANTANDER,54800,TEORAMA,RURAL,MASCULINO,ARMA DE FUEGO,ENFRENTAMIENTO ENTRE SUBVERSIVOS,HOMICIDIO INTENCIONAL O CON DOLO,1,19
1,23/01/2025,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1,11
2,02/11/2025,76,VALLE DEL CAUCA,76001,CALI,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1,11
3,16/01/2025,54,NORTE DE SANTANDER,54810,TIBU,RURAL,MASCULINO,ARMA DE FUEGO,ENFRENTAMIENTO ENTRE SUBVERSIVOS,HOMICIDIO INTENCIONAL O CON DOLO,1,10
4,17/09/2025,76,VALLE DEL CAUCA,76001,CALI,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1,8
5,23/04/2025,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1,7
6,08/12/2025,76,VALLE DEL CAUCA,76001,CALI,URBANA,MASCULINO,ARMA DE FUEGO,SICARIATO,HOMICIDIO INTENCIONAL O CON DOLO,1,7
7,25/12/2025,11,BOGOTA D.C.,11001,BOGOTA D.C.,URBANA,MASCULINO,ARMA BLANCA / CORTOPUNZANTE,RIÑAS,HOMICIDIO INTENCIONAL O CON DOLO,1,7
8,18/01/2025,95,GUAVIARE,95015,CALAMAR,RURAL,MASCULINO,ARMA DE FUEGO,ENFRENTAMIENTO ENTRE CRIMEN ORGANIZADO,HOMICIDIO INTENCIONAL O CON DOLO,1,7
9,14/01/2025,23,CORDOBA,23466,MONTELIBANO,RURAL,MASCULINO,ARMA DE FUEGO,ENFRENTAMIENTO ELN-CRIMEN ORGANIZADO,HOMICIDIO INTENCIONAL O CON DOLO,1,7


**Medellín: dos filtros, ¿una cifra?**

In [ ]:
# Celda 15 · ¿Cuántos hechos ocurrieron en Medellín para el año 2025? Filtrar por la columna MUNICIPIO.
sql("""
SELECT MUNICIPIO, COUNT (*) AS filas
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31' AND "MUNICIPIO" = 'MEDELLIN'
GROUP BY MUNICIPIO
""")

,MUNICIPIO,filas
0,MEDELLIN,332


In [ ]:
# Celda 16 · ¿Cuántos hechos ocurrieron en Medellín para el año 2025? Filtrar por la columna COD_MUNI.
sql("""
SELECT COUNT (*) AS filas
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31' AND "COD_MUNI" = '5001'
""")

,filas
0,332


In [ ]:
# Celda 17 · ¿Cuántas víctimas de homicidio hubo en Medellín para el año 2025? Filtrar por la columna MUNICIPIO.
sql("""
SELECT SUM(CANTIDAD) AS cantidad_total
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31' AND "MUNICIPIO" = 'MEDELLIN'
""")

,cantidad_total
0,333.0


In [ ]:
# Celda 18 · ¿Cuántas víctimas de homicidio hubo en Medellín para el año 2025? Filtrar por la columna COD_MUNI.
sql("""
SELECT SUM(CANTIDAD) AS cantidad_total
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31' AND "COD_MUNI" = '5001'
""")

,cantidad_total
0,333.0


In [ ]:
# Celda 19 · ¿Cuántas víctimas de homicidio hubo en Medellín para el año 2025 discriminando por zona rural y urbana?
sql("""
SELECT ZONA, SUM(CANTIDAD) AS total_victimas
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
  AND "COD_MUNI" = '5001'
GROUP BY ZONA
""")

,ZONA,total_victimas
0,URBANA,319.0
1,RURAL,14.0


**¿Categoría o dato que falta?**

In [ ]:
# Celda 20 · ¿Cuántas víctimas de homicidio agrupados por 'Modalidad Presunta' y 'Sexo' hubo en 2025?
sql("""
SELECT " MODALIDAD PRESUNTA", SEXO, COUNT(*) AS total_casos
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY " MODALIDAD PRESUNTA", SEXO
ORDER BY " MODALIDAD PRESUNTA" ASC, total_casos DESC
""")

,MODALIDAD PRESUNTA,SEXO,total_casos
0,ATACADOS POR GAO O GAO-R,MASCULINO,617
1,ATACADOS POR GAO O GAO-R,FEMENINO,52
2,ATRACO,MASCULINO,510
3,ATRACO,FEMENINO,38
4,DESMEMBRAMIENTO,MASCULINO,23
5,ENFRENTAMIENTO ELN-CRIMEN ORGANIZADO,MASCULINO,47
6,ENFRENTAMIENTO ELN-CRIMEN ORGANIZADO,FEMENINO,4
7,ENFRENTAMIENTO ENTRE CRIMEN ORGANIZADO,MASCULINO,39
8,ENFRENTAMIENTO ENTRE CRIMEN ORGANIZADO,FEMENINO,9
9,ENFRENTAMIENTO ENTRE DELINCUENCIA COMUN,MASCULINO,13


In [ ]:
# Celda 21 · ¿Se cuenta con todos los meses entre nov/2025 y oct/2026?
sql("""
SELECT
    SUBSTR("FECHA HECHO", 4, 7) AS mes_anio,
    COUNT(*) AS total_casos
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2025-11-01' AND '2026-10-31'
GROUP BY mes_anio
ORDER BY strptime('01/' || mes_anio, '%d/%m/%Y') ASC
""")

,mes_anio,total_casos
0,11/2025,1192
1,12/2025,1259
2,01/2026,1197
3,02/2026,1089
4,03/2026,1180
5,04/2026,1161
6,05/2026,1181
7,06/2026,1145
8,07/2026,1257
9,08/2026,1195


In [ ]:
# Celda 22 · ¿Se cuenta con todos los meses entre nov/2024 y oct/2025?
sql("""
SELECT
    SUBSTR("FECHA HECHO", 4, 7) AS mes_anio,
    COUNT(*) AS total_casos
FROM datos
WHERE strptime("FECHA HECHO", '%d/%m/%Y') BETWEEN '2024-11-01' AND '2025-10-31'
GROUP BY mes_anio
ORDER BY strptime('01/' || mes_anio, '%d/%m/%Y') ASC
""")

,mes_anio,total_casos
0,11/2024,1154
1,12/2024,1238
2,01/2025,1207
3,02/2025,945
4,03/2025,1178
5,04/2025,1076
6,05/2025,1206
7,06/2025,1191
8,07/2025,1175
9,08/2025,1235


## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.